# Just-in-Time (JIT) compilation with Numba and JAX

**Python for HPC course**

Max Planck Computing and Data Facility, Garching

## Just-in-Time (JIT) compilation
* Plain Python code is typically interpreted which makes execution much slower compared to compiled code, such as C or Fortran
* JIT: compilation of a program or function at *runtime* when the datatypes of variables (function parameters, arrays) and their attributes (size and dimension of arrays) are known
    * initial overhead is caused once by compilation
    * subsequent calls then use cached compiled code
* Java or Julia are examples for languages that natively make use of JIT by default
* JIT for Python is available through packages such as `Numba`, `JAX`, `numexpr`

## Numba
* Numba is a Just-in-Time (JIT) compiler for Python code
* designed to accelerate numerical computation implemented with NumPy
* convenient usage via function decorators (`@jit`), not via a separate language like Cython  
  $\to$ keeps Python code compatibility
* based on the LLVM compiler framework, support for
    * CPUs
    * GPUs (NVIDIA, AMD)
* Documenation: https://numba.pydata.org/

### Code which will potentially benefit from Numba

Numerical computations using

* NumPy expressions in array notation
* explicit loops accessing NumPy arrays element-wise

... on sufficiently large data!

### Code which likely won't benefit

* calls to packages beyond plain Python and NumPy
* IO-intense code

### Numba on explicit loops

Examples in the following are adapted from  
https://numba.readthedocs.io/en/stable/user/5minguide.html

In [39]:
# plain NumPy example with explicit loops
import numpy as np

m = 512
x = np.arange(m*m, dtype=np.float64).reshape(m, m)

def trace_add(a):
    trace = 0.0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i, i])
    b = a + trace
    return b

%timeit trace_add(x)

666 μs ± 64.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [40]:
# plain NumPy example with explicit loops AND Numba @jit
import numpy as np
from numba import jit

m = 512
x = np.arange(m*m, dtype=np.float64).reshape(m, m)

@jit
def trace_add(a):
    trace = 0.0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i, i])
    b = a + trace
    return b
trace_add(x)

%timeit trace_add(x)

134 μs ± 39.2 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Under the hood of Numba

When a `@jit`-decorated function is called for the first time, Numba will

1. read the Python bytecode of the function
2. consider information about the actual function arguments (datatype, array sizes, etc.)
3. analyze code, create LLVM-internal representation, optimize
4. generate machine code version using LLVM

The compiled version is cached and then used each time the function is called.  
$\to$ For performance benchmarks, one should exclude the first call.

### Numba's `nopython` and `object` modes

* The decorator `@jit` instructs Numba to compile the decorated function such that is runs entirely without the Python interpreter (is equivalent to legacy `@jit(nopython=True)` / `@njit`).
* With `@jit` the compilation fails in case callbacks to Python would happen
* Use `@jit(forceobj=True)` to explicitly allow callbacks to Python
* Use `@jit(forceobj=True, looplift=True)` to try to compile the loop arithmetic for potential gain

### Numba on NumPy expressions
* Numba is able to compile NumPy expressions using array notation ("vectorization")

In [41]:
import numpy as np
from numba import jit

@jit
def ident_array(x):
    """Numpy expression example, array notation"""
    return np.cos(x)**2 + np.sin(x)**2

@jit
def ident_loop(x):
    """Numpy expression example, loop version"""
    r = np.empty_like(x)
    n = len(x)
    for i in range(n):
        r[i] = np.cos(x[i])**2 + np.sin(x[i])**2
    return r

In [42]:
x = np.arange(2**20, dtype=np.float32)
ident_array(x); ident_loop(x);

%timeit ident_array(x)
%timeit ident_loop(x)

9.3 ms ± 1.15 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
8.48 ms ± 4.13 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Numba's `fastmath` mode

* `fastmath=True` enables compiler optimizations and library calls which *might* further speed up your code
* most speedup to be expected for transcendental functions (sin, cos, exp, log, etc.)
* relaxes IEEE 754 requirements on floating point arithmetic
* potential to alter numerical results $\to$ check results carefully
* similar to well-known GCC ('-ffash-math') or Intel compiler flags ('-fp-model fast=2')

In [43]:
import numpy as np
from numba import jit

@jit
def ident_slowmath(x):
    np.sum(np.log(np.exp(x)) ** 2)

@jit(fastmath=True)
def ident_fastmath(x):
    np.sum(np.log(np.exp(x)) ** 2)

In [44]:
x = np.ones(2**20, dtype=np.float32)
ident_slowmath(x); ident_fastmath(x);

%timeit ident_slowmath(x)
%timeit ident_fastmath(x)

9.76 ms ± 1.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
290 μs ± 112 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Multithreading with Numba

* auto-parallelization of NumPy expressions
* explicit loop parallelization using `prange()`
* number of threads
    * all available cores (default)
    * set via environment variable `NUMBA_NUM_THREADS`
    * set via `numba.set_num_threads()`
* available backends (selected automatically)
    * Numba-internal workqueue
    * Intel TBB
    * OpenMP

In [45]:
import numpy as np
from numba import jit, prange

@jit
def ident_array_seq(x):
    return np.cos(x)**2 + np.sin(x)**2

@jit(parallel=True)
def ident_array_par(x):
    return np.cos(x)**2 + np.sin(x)**2

@jit(parallel=True)
def ident_loop_par(x):
    r = np.empty_like(x)
    n = len(x)
    for i in prange(n):
        r[i] = np.cos(x[i])**2 + np.sin(x[i])**2
    return r

In [46]:
x = np.arange(2**20, dtype=np.float32)
ident_array_seq(x); ident_array_par(x); ident_loop_par(x)

%timeit ident_array_seq(x)
%timeit ident_array_par(x)
%timeit ident_loop_par(x)

8.47 ms ± 2.62 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.7 ms ± 181 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.14 ms ± 2.11 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Advanced Numba features of potential interest

* `@vectorize`, generate NumPy `ufunc` objects
* `@stencil`, define stencil operations on NumPy arrays
* `@cuda`, target NVIDIA CUDA GPUs
* `@roc`, target AMD GPUs

Please see the documentation for further information.  
https://numba.readthedocs.io/en/stable/

## JAX
* JAX is a library for numerical computing and machine learning applications
* JAX provides a NumPy-like API which can be used as a drop-in replacement for original NumPy
* JAX provides just-in-Time (JIT) compilation and automatic differentiation
* built on the [XLA](https://www.tensorflow.org/xla) compiler with support for
    * CPUs
    * GPUs (NVIDIA, AMD)
    * Google TPUs
* https://jax.readthedocs.io/en/latest/

## JAX vs. Numba

* for JIT, both provide a simple `jit` interface
* both use LLVM as the compiler backend; jax with a layer of indirection via the XLA compiler
* JAX supports CPU/GPU/TPU backends with exactly the same codebase (e.g., no CUDA knowledge necessary)
* JAX performs well when used on accelerator devices
* JAX will fail on code it doesn't know
* JAX provides automated differentiation in addition
* JAX is general purpose but many concepts have a background in machine learning (e.g. batch parallelization)

### JIT compilation with JAX

In [1]:
from jax import jit
import jax.numpy as np

m = 512
x = np.arange(m*m, dtype=np.float32).reshape(m, m)

@jit
def trace_add(a):
    trace = 0.0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i, i])
    b = a + trace
    return b
trace_add(x).block_until_ready()   # JIT at first call

%timeit trace_add(x).block_until_ready()

164 μs ± 4.72 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Autodifferentiation with JAX

In [48]:
from jax import grad

def reduce_trace_add(a):
    """Reduction to have scalar output."""
    return trace_add(a).sum()

grad_reduce_trace_add = jit(grad(reduce_trace_add))
grad_reduce_trace_add(x)

%timeit grad_reduce_trace_add(x)

54.1 ms ± 60.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Additional features of JAX

* auto-differentiation via `grad`
* multivariate differentiation via `jacfwd`, `jacrev`, `jvp`, `vjp`
* support for vectorization and parallelization: `vmap`, `pmap`
* JAX is still under active development but becoming increasingly [popular](https://github.com/n2cholas/awesome-jax)

## Other JIT solutions

### NumExpr: Fast numerical expression evaluator for NumPy

* translates (nested) NumPy expressions into loop representation and compiles  
  $\rightarrow$ memory saving, cache blocking, thread parallelization, SIMD
* caveat: NumPy expressions must be passed as strings
* https://github.com/pydata/numexpr

### *pystencils*

* sympy-based code generator for **stencil computations on NumPy arrays**
* due to knowledge about the structure of the stencil, the code can be highly optimized
* https://i10git.cs.fau.de/pycodegen/pystencils

Optional exercise: Implement the diffusion case study using these tools!